# Optimizer

Rule-based logical plan optimizer for MXFrame.

In [ ]:
#| default_exp optimizer

In [ ]:
#| export
from dataclasses import dataclass
from typing import List, Tuple

from mxframe.lazy_expr import Expr
from mxframe.lazy_frame import (
    LogicalPlan,
    Scan,
    Filter,
    Project,
    Aggregate,
    Sort,
    Limit,
    Distinct,
    Join,
)


__all__ = ["OptimizationResult", "PlanOptimizer", "optimize_plan"]


@dataclass
class OptimizationResult:
    """Optimizer output containing a potentially rewritten plan and pass trace."""

    plan: LogicalPlan
    trace: List[str]


class PlanOptimizer:
    """Rule-based optimizer for LogicalPlan trees.

    This starts with conservative semantic-preserving rewrites.
    """

    def optimize(self, plan: LogicalPlan) -> OptimizationResult:
        current = plan
        trace: List[str] = []

        current, changed = self._merge_adjacent_filters(current)
        if changed:
            trace.append("merge_adjacent_filters")

        current, changed = self._collapse_nested_limits(current)
        if changed:
            trace.append("collapse_nested_limits")

        current, changed = self._normalize_sort_flags(current)
        if changed:
            trace.append("normalize_sort_flags")

        current, changed = self._remove_identity_projects(current)
        if changed:
            trace.append("remove_identity_projects")

        if not trace:
            trace.append("no_op")

        return OptimizationResult(plan=current, trace=trace)

    def _merge_adjacent_filters(self, plan: LogicalPlan) -> Tuple[LogicalPlan, bool]:
        changed = False

        def walk(node: LogicalPlan) -> LogicalPlan:
            nonlocal changed
            if isinstance(node, Filter):
                child = walk(node.input)
                if isinstance(child, Filter):
                    changed = True
                    merged = Expr("and", child.predicate, node.predicate)
                    return Filter(child.input, merged)
                return Filter(child, node.predicate)
            if isinstance(node, Project):
                return Project(walk(node.input), node.exprs)
            if isinstance(node, Aggregate):
                return Aggregate(walk(node.input), node.group_by, node.aggs)
            if isinstance(node, Sort):
                return Sort(walk(node.input), node.by, node.descending)
            if isinstance(node, Limit):
                return Limit(walk(node.input), node.n)
            if isinstance(node, Distinct):
                return Distinct(walk(node.input), node.subset)
            if isinstance(node, Join):
                return Join(
                    walk(node.left),
                    walk(node.right),
                    node.left_on,
                    node.right_on,
                    node.how,
                )
            return node

        return walk(plan), changed

    def _collapse_nested_limits(self, plan: LogicalPlan) -> Tuple[LogicalPlan, bool]:
        changed = False

        def walk(node: LogicalPlan) -> LogicalPlan:
            nonlocal changed
            if isinstance(node, Limit):
                child = walk(node.input)
                if isinstance(child, Limit):
                    changed = True
                    return Limit(child.input, min(child.n, node.n))
                return Limit(child, node.n)
            if isinstance(node, Filter):
                return Filter(walk(node.input), node.predicate)
            if isinstance(node, Project):
                return Project(walk(node.input), node.exprs)
            if isinstance(node, Aggregate):
                return Aggregate(walk(node.input), node.group_by, node.aggs)
            if isinstance(node, Sort):
                return Sort(walk(node.input), node.by, node.descending)
            if isinstance(node, Distinct):
                return Distinct(walk(node.input), node.subset)
            if isinstance(node, Join):
                return Join(
                    walk(node.left),
                    walk(node.right),
                    node.left_on,
                    node.right_on,
                    node.how,
                )
            return node

        return walk(plan), changed

    def _normalize_sort_flags(self, plan: LogicalPlan) -> Tuple[LogicalPlan, bool]:
        changed = False

        def walk(node: LogicalPlan) -> LogicalPlan:
            nonlocal changed
            if isinstance(node, Sort):
                child = walk(node.input)
                desc = list(node.descending)
                if len(desc) == 1 and len(node.by) > 1:
                    changed = True
                    desc = desc * len(node.by)
                elif len(desc) != len(node.by):
                    changed = True
                    desc = (desc + [False] * len(node.by))[: len(node.by)]
                return Sort(child, node.by, desc)
            if isinstance(node, Filter):
                return Filter(walk(node.input), node.predicate)
            if isinstance(node, Project):
                return Project(walk(node.input), node.exprs)
            if isinstance(node, Aggregate):
                return Aggregate(walk(node.input), node.group_by, node.aggs)
            if isinstance(node, Limit):
                return Limit(walk(node.input), node.n)
            if isinstance(node, Distinct):
                return Distinct(walk(node.input), node.subset)
            if isinstance(node, Join):
                return Join(
                    walk(node.left),
                    walk(node.right),
                    node.left_on,
                    node.right_on,
                    node.how,
                )
            return node

        return walk(plan), changed

    def _remove_identity_projects(self, plan: LogicalPlan) -> Tuple[LogicalPlan, bool]:
        changed = False

        def walk(node: LogicalPlan) -> LogicalPlan:
            nonlocal changed
            if isinstance(node, Project):
                child = walk(node.input)
                if isinstance(child, Scan) and self._is_identity_scan_project(node, child):
                    changed = True
                    return child
                return Project(child, node.exprs)
            if isinstance(node, Filter):
                return Filter(walk(node.input), node.predicate)
            if isinstance(node, Aggregate):
                return Aggregate(walk(node.input), node.group_by, node.aggs)
            if isinstance(node, Sort):
                return Sort(walk(node.input), node.by, node.descending)
            if isinstance(node, Limit):
                return Limit(walk(node.input), node.n)
            if isinstance(node, Distinct):
                return Distinct(walk(node.input), node.subset)
            if isinstance(node, Join):
                return Join(
                    walk(node.left),
                    walk(node.right),
                    node.left_on,
                    node.right_on,
                    node.how,
                )
            return node

        return walk(plan), changed

    @staticmethod
    def _is_identity_scan_project(project: Project, scan: Scan) -> bool:
        cols = list(scan.table.column_names)
        if len(project.exprs) != len(cols):
            return False
        for idx, expr in enumerate(project.exprs):
            if expr.op != "col":
                return False
            if not expr.args or expr.args[0] != cols[idx]:
                return False
            if getattr(expr, "_alias", None) is not None:
                return False
        return True


def optimize_plan(plan: LogicalPlan) -> OptimizationResult:
    """Optimize a LogicalPlan with default pass pipeline."""
    return PlanOptimizer().optimize(plan)